# Extraction automatique des données de pathologies — data.ameli.fr

Ce notebook interroge directement l'**API ouverte** du site
[data.ameli.fr](https://data.ameli.fr/pages/pathologies) pour reconstruire,
pour **toutes les pathologies disponibles**, un tableau équivalent à celui
que vous avez commencé à la main :

| Groupe | Sous-groupe | Prévalence (%) | Dépenses totales | Répartition femmes (%) | Tranche d'âge la plus touchée |

**Comment ça marche ?**
data.ameli.fr est construit sur la plateforme *OpenDataSoft*. Les données
sont donc accessibles sans clé ni authentification via deux jeux de
données publics :

- `effectifs` → prévalence, sexe, tranche d'âge, par pathologie
- `depenses` → dépenses remboursées par pathologie

⚠️ Les noms de colonnes exacts du jeu `effectifs` n'ont pas pu être
vérifiés à 100% avant l'exécution : ce notebook **affiche systématiquement
les colonnes réelles** avant de les utiliser, pour éviter de calculer
silencieusement un résultat faux avec un mauvais nom de colonne. Si les
noms diffèrent légèrement de ceux prévus, ajustez simplement la variable
correspondante dans la cellule de configuration (section 3).


In [1]:
# 1. Imports
import io
import re
import requests
import pandas as pd

pd.set_option("display.max_columns", 100)


## 2. Téléchargement des jeux de données

On utilise le point d'export CSV de l'API Explore v2.1 d'OpenDataSoft,
qui ne nécessite aucune clé d'API et n'est pas limité en volume :

```
https://data.ameli.fr/api/explore/v2.1/catalog/datasets/{dataset_id}/exports/csv
```


In [2]:
BASE_URL = "https://data.ameli.fr/api/explore/v2.1/catalog/datasets/{dataset_id}/exports/csv"

def download_dataset(dataset_id: str) -> pd.DataFrame:
    """Télécharge un jeu de données data.ameli.fr en entier (export CSV)."""
    url = BASE_URL.format(dataset_id=dataset_id)
    params = {
        "delimiter": ";",       # séparateur utilisé par l'export ODS
        "list_separator": ",",
        "use_labels": "true",   # libellés lisibles plutôt que codes internes
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), sep=";")
    return df

print("Téléchargement de 'effectifs' (prévalence, sexe, âge)...")
df_effectifs = download_dataset("effectifs")
print(df_effectifs.shape)

print("Téléchargement de 'depenses' (dépenses par pathologie)...")
df_depenses = download_dataset("depenses")
print(df_depenses.shape)


Téléchargement de 'effectifs' (prévalence, sexe, âge)...
(5796000, 16)
Téléchargement de 'depenses' (dépenses par pathologie)...
(24800, 14)


## 3. Exploration des colonnes (à vérifier avant de continuer)

On regarde les vraies colonnes renvoyées par l'API, ainsi qu'un aperçu des
valeurs, pour identifier précisément :
- la colonne "année"
- les colonnes de hiérarchie pathologie (`patho_niv1`, `patho_niv2`, ...)
- la colonne de territoire (pour ne garder que la France entière)
- pour `effectifs` : la colonne sexe, la colonne tranche d'âge, la colonne
  de prévalence et la colonne d'effectif


In [3]:
print("Colonnes de df_depenses :")
print(list(df_depenses.columns))
df_depenses.head(3)


Colonnes de df_depenses :
['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'dep_niv_1', 'dep_niv_2', 'montant', 'Ntop', 'N_recourant_au_poste', 'montant_moy', 'Niveau prioritaire', 'tri', 'type_somme']


,annee,patho_niv1,patho_niv2,patho_niv3,top,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy,Niveau prioritaire,tri,type_somme
0,2018,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Actes et consultations externes MCO secteur pu...,130609573,1730020.0,1005620.0,75.0,"1,2,3",16.0,Partiel
1,2018,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Hospitalisations en HAD secteur privé remboursées,377106,1730020.0,6030.0,0.0,"1,2,3",16.0,Partiel
2,2018,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Hospitalisations séjour MCO secteur privé remb...,18723264,1730020.0,322040.0,11.0,"1,2,3",16.0,Partiel


In [4]:
print("Colonnes de df_effectifs :")
print(list(df_effectifs.columns))
df_effectifs.head(3)


Colonnes de df_effectifs :
['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'cla_age_5', 'sexe', 'region', 'dept', 'Ntop', 'Npop', 'prev', 'Niveau prioritaire', 'libelle_classe_age', 'libelle_sexe', 'tri']


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri
0,2017,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,00-04,1,1,971,200.0,10210,1.949,"1,2,3",de 0 à 4 ans,hommes,16.0
1,2017,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,00-04,1,2,999,140.0,9120,1.481,"1,2,3",de 0 à 4 ans,hommes,16.0
2,2017,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,00-04,1,6,976,100.0,7930,1.311,"1,2,3",de 0 à 4 ans,hommes,16.0


## 4. Configuration — à ajuster si besoin

Renseignez ici les noms de colonnes **tels qu'affichés ci-dessus**.
Des valeurs par défaut raisonnables sont proposées ; si l'exécution de la
section 5 échoue avec un `KeyError`, corrigez simplement les noms ci-dessous.


In [5]:
ANNEE = 2024

# --- Colonnes communes (hiérarchie pathologie) ---
COL_GROUPE = "patho_niv1"      # groupe de pathologies (ex: Cancers)
COL_SOUS_GROUPE = "patho_niv2" # sous-groupe (ex: Sein, Prostate...)

# --- Jeu "depenses" ---
COL_ANNEE_DEP = "annee"
COL_MONTANT = "montant"

# --- Jeu "effectifs" ---
COL_ANNEE_EFF = "annee"
# Ces 3 noms sont des hypothèses à confirmer avec la liste imprimée en section 3
COL_PREVALENCE = next((c for c in df_effectifs.columns if "prevalence" in c.lower() or "prev" == c.lower()), "prevalence")
COL_SEXE = next((c for c in df_effectifs.columns if "sexe" in c.lower()), "sexe")
COL_AGE = next((c for c in df_effectifs.columns if "age" in c.lower() or "âge" in c.lower()), "classe_age")
COL_EFFECTIF = next((c for c in df_effectifs.columns if "npop" in c.lower() or "effectif" in c.lower() or c.lower().startswith("n_")), "npop")
COL_TERRITOIRE = next((c for c in df_effectifs.columns if "dep_niv" in c.lower() or "territoire" in c.lower()), None)

print("Colonnes détectées automatiquement pour 'effectifs':")
print("prévalence  ->", COL_PREVALENCE)
print("sexe        ->", COL_SEXE)
print("âge         ->", COL_AGE)
print("effectif    ->", COL_EFFECTIF)
print("territoire  ->", COL_TERRITOIRE)


Colonnes détectées automatiquement pour 'effectifs':
prévalence  -> prev
sexe        -> sexe
âge         -> cla_age_5
effectif    -> Npop
territoire  -> None


## 5. Filtrage sur l'année 2024 (et sur la France entière si un niveau territorial existe)


In [6]:
dep_2024 = df_depenses[df_depenses[COL_ANNEE_DEP] == ANNEE].copy()

eff_2024 = df_effectifs[df_effectifs[COL_ANNEE_EFF] == ANNEE].copy()

if COL_TERRITOIRE and COL_TERRITOIRE in eff_2024.columns:
    print("Valeurs uniques de la colonne territoire :", eff_2024[COL_TERRITOIRE].unique()[:20])
    # On essaie de garder uniquement le niveau national (France entière)
    niveau_national = [v for v in eff_2024[COL_TERRITOIRE].dropna().unique() if "france" in str(v).lower()]
    if niveau_national:
        eff_2024 = eff_2024[eff_2024[COL_TERRITOIRE] == niveau_national[0]]
        print("Filtré sur le territoire :", niveau_national[0])

print(dep_2024.shape, eff_2024.shape)


(2480, 14) (579600, 16)


## 6. Dépenses totales par groupe / sous-groupe de pathologies


In [7]:
depenses_totales = (
    dep_2024.groupby([COL_GROUPE, COL_SOUS_GROUPE])[COL_MONTANT]
    .sum()
    .reset_index()
    .rename(columns={COL_MONTANT: "Dépenses totales en 2024"})
)
depenses_totales.head()


,patho_niv1,patho_niv2,Dépenses totales en 2024
0,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,12695672606
1,Cancers,Autres cancers,93519992762
2,Cancers,Cancer bronchopulmonaire,22083511545
3,Cancers,Cancer colorectal,12133445559
4,Cancers,Cancer de la prostate,18226274864


## 7. Prévalence par groupe / sous-groupe (population totale, tous sexes confondus)

On cherche, dans la colonne sexe, la modalité correspondant à "ensemble"
(hommes + femmes) et dans la colonne âge la modalité correspondant à
"tous âges", afin de récupérer la prévalence globale par pathologie.


In [8]:
print("Modalités de la colonne sexe :", eff_2024[COL_SEXE].unique())
print("Modalités de la colonne âge (extrait) :", eff_2024[COL_AGE].unique()[:20])


Modalités de la colonne sexe : [2 9 1]
Modalités de la colonne âge (extrait) : ['15-19' '20-24' '25-29' '30-34' '35-39' '40-44' '45-49' '50-54' '55-59'
 '60-64' '65-69' '70-74' '75-79' '80-84' '85-89' '90-94' '95et+' 'tsage'
 '00-04' '05-09']


In [9]:
# Modalité "ensemble" pour le sexe et "tous âges" pour l'âge : à ajuster
# selon ce qui est imprimé juste au-dessus si besoin.
SEXE_ENSEMBLE = next((v for v in eff_2024[COL_SEXE].unique() if "ensemble" in str(v).lower() or "tous" in str(v).lower()), eff_2024[COL_SEXE].unique()[0])
AGE_TOUS = next((v for v in eff_2024[COL_AGE].unique() if "tous" in str(v).lower() or "ensemble" in str(v).lower()), None)

base_prevalence = eff_2024[eff_2024[COL_SEXE] == SEXE_ENSEMBLE]
if AGE_TOUS is not None:
    base_prevalence = base_prevalence[base_prevalence[COL_AGE] == AGE_TOUS]

prevalence = (
    base_prevalence.groupby([COL_GROUPE, COL_SOUS_GROUPE])[COL_PREVALENCE]
    .mean()
    .reset_index()
    .rename(columns={COL_PREVALENCE: "Prévalence (%) en 2024"})
)
prevalence.head()


,patho_niv1,patho_niv2,Prévalence (%) en 2024
0,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,4.088211
1,Cancers,Autres cancers,2.730621
2,Cancers,Cancer bronchopulmonaire,0.308498
3,Cancers,Cancer colorectal,0.774220
4,Cancers,Cancer du sein de la femme,2.301747


## 8. Répartition femmes / hommes (%) par groupe / sous-groupe


In [ ]:
repartition = eff_2024[eff_2024[COL_SEXE] != SEXE_ENSEMBLE].copy()
if AGE_TOUS is not None:
    repartition = repartition[repartition[COL_AGE] == AGE_TOUS]

pivot_sexe = (
    repartition.groupby([COL_GROUPE, COL_SOUS_GROUPE, COL_SEXE])[COL_EFFECTIF]
    .sum()
    .unstack(COL_SEXE)
)

col_femmes = next((c for c in pivot_sexe.columns if "fem" in str(c).lower()), None)

if col_femmes is not None:
    pivot_sexe["Répartition des effectifs en 2024 (femmes)"] = (
        pivot_sexe[col_femmes] / pivot_sexe.sum(axis=1) * 100
    ).round(2).astype(str) + "%"
    repartition_femmes = pivot_sexe[["Répartition des effectifs en 2024 (femmes)"]].reset_index()
else:
    repartition_femmes = pivot_sexe.reset_index()

repartition_femmes.head()


sexe,patho_niv1,patho_niv2,1,9
0,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,194138730.0,4.047482e+08
1,Cancers,Autres cancers,582416190.0,1.214245e+09
2,Cancers,Cancer bronchopulmonaire,582416190.0,1.214245e+09
3,Cancers,Cancer colorectal,582416190.0,1.214245e+09
4,Cancers,Cancer de la prostate,582416190.0,6.365560e+08


## 9. Tranche d'âge où la prévalence est la plus élevée


In [11]:
base_age = eff_2024[eff_2024[COL_SEXE] == SEXE_ENSEMBLE].copy()
if AGE_TOUS is not None:
    base_age = base_age[base_age[COL_AGE] != AGE_TOUS]

idx_max = base_age.groupby([COL_GROUPE, COL_SOUS_GROUPE])[COL_PREVALENCE].idxmax()
tranche_max = (
    base_age.loc[idx_max, [COL_GROUPE, COL_SOUS_GROUPE, COL_AGE]]
    .rename(columns={COL_AGE: "Tranche d'âge où la prévalence est la plus élevée"})
)
tranche_max.head()


,patho_niv1,patho_niv2,Tranche d'âge où la prévalence est la plus élevée
3809791,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,95et+
1668949,Cancers,Autres cancers,95et+
3802836,Cancers,Cancer bronchopulmonaire,75-79
1826422,Cancers,Cancer colorectal,85-89
5789730,Cancers,Cancer du sein de la femme,85-89


## 10. Assemblage du tableau final


In [12]:
tableau_final = (
    prevalence
    .merge(depenses_totales, on=[COL_GROUPE, COL_SOUS_GROUPE], how="outer")
    .merge(repartition_femmes, on=[COL_GROUPE, COL_SOUS_GROUPE], how="outer")
    .merge(tranche_max, on=[COL_GROUPE, COL_SOUS_GROUPE], how="outer")
    .rename(columns={COL_GROUPE: "Groupes de pathologies", COL_SOUS_GROUPE: "Sous-groupes de pathologies"})
    .sort_values(["Groupes de pathologies", "Sous-groupes de pathologies"])
    .reset_index(drop=True)
)

tableau_final


,Groupes de pathologies,Sous-groupes de pathologies,Prévalence (%) en 2024,Dépenses totales en 2024,1,9,Tranche d'âge où la prévalence est la plus élevée
0,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,4.088211,12695672606,194138730.0,4.047482e+08,95et+
1,Cancers,Autres cancers,2.730621,93519992762,582416190.0,1.214245e+09,95et+
2,Cancers,Cancer bronchopulmonaire,0.308498,22083511545,582416190.0,1.214245e+09,75-79
3,Cancers,Cancer colorectal,0.774220,12133445559,582416190.0,1.214245e+09,85-89
4,Cancers,Cancer de la prostate,NaN,18226274864,582416190.0,6.365560e+08,NaN
5,Cancers,Cancer du sein de la femme,2.301747,30101316704,NaN,6.650108e+08,85-89
6,Diabète,Diabète,7.795435,34506896324,194138730.0,4.047482e+08,80-84
7,Hospitalisation pour Covid-19,Hospitalisation pour Covid-19,0.463841,1309409470,194138730.0,4.047482e+08,95et+
8,Hospitalisations hors pathologies repérées (av...,Hospitalisations hors pathologies repérées (av...,17.578811,141370462278,194138730.0,4.047482e+08,95et+
9,Insuffisance rénale chronique terminale,Dialyse chronique,0.151800,11649791793,194138730.0,4.047482e+08,80-84


## 11. Export du résultat


In [13]:
tableau_final.to_csv("pathologies_2024_complet.csv", index=False, sep=";")
tableau_final.to_excel("pathologies_2024_complet.xlsx", index=False)
print("Fichiers exportés : pathologies_2024_complet.csv / .xlsx")


Fichiers exportés : pathologies_2024_complet.csv / .xlsx


## Notes

- Certains groupes de pathologies (ex : *maladies cardio-neurovasculaires*)
  n'ont pas systématiquement de sous-groupe dans le jeu `depenses` — dans
  ce cas, `Sous-groupes de pathologies` peut être identique au groupe, ou
  vide selon la structure réelle des données.
- Le secret statistique s'applique : en dessous de 11 patients, l'API
  renvoie `"NS"` (non significatif) — ces valeurs se retrouveront telles
  quelles dans le tableau.
- Si une cellule échoue avec un `KeyError` ou un résultat vide, relancez
  la section 3 pour re-vérifier les noms de colonnes réels, puis ajustez
  la section 4 en conséquence.
